In [2]:
import os

# Configuración de carpetas destino
BASE_DIR = "."  # Directorio actual
REAL_MINI_DIR = os.path.join(BASE_DIR, "Real_mini")
FAKE_MINI_DIR = os.path.join(BASE_DIR, "Fake_mini")

VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff')
CATEGORIES = ["deepfakes", "face2face", "faceshifter", "faceswap", "neuraltextures"]

TARGET_TOTAL = 100  # Tamaño final deseado para la mini-mini tupla

registros_totales = []

# -------------------------------------------------------------
# 1. Obtener todos los registros reales
# -------------------------------------------------------------
if os.path.exists(REAL_MINI_DIR):
    real_files = sorted([f for f in os.listdir(REAL_MINI_DIR) if f.lower().endswith(VALID_EXTENSIONS)])
    for filename in real_files:
        file_path = os.path.abspath(os.path.join(REAL_MINI_DIR, filename))
        registros_totales.append((0, 'real', file_path))

# -------------------------------------------------------------
# 2. Obtener todos los registros fakes
# -------------------------------------------------------------
if os.path.exists(FAKE_MINI_DIR):
    fake_files = sorted([f for f in os.listdir(FAKE_MINI_DIR) if f.lower().endswith(VALID_EXTENSIONS)])
    for filename in fake_files:
        file_path = os.path.abspath(os.path.join(FAKE_MINI_DIR, filename))
        
        cat_encontrada = 'unknown'
        filename_lower = filename.lower()
        for cat in CATEGORIES:
            if filename_lower.startswith(cat):
                cat_encontrada = cat
                break
                
        registros_totales.append((1, cat_encontrada, file_path))

# -------------------------------------------------------------
# 3. Muestreo estratificado manteniendo la distribución original
# -------------------------------------------------------------
total_elementos = len(registros_totales)

if total_elementos == 0:
    print("❌ No se encontraron imágenes para procesar.")
    dataset_tuple = ()
else:
    # Agrupar registros por su categoría (tipo_fake o 'real')
    grupos = {}
    for item in registros_totales:
        cat = item[1]
        grupos.setdefault(cat, []).append(item)

    registros_seleccionados = []
    
    print(f"📊 Distribución original ({total_elementos} total):")
    
    for cat, items in grupos.items():
        # Calcular la proporción de esta categoría respecto al total
        proporcion = len(items) / total_elementos
        # Determinar cuántas imágenes le corresponden en la muestra de 100
        cant_objetivo = max(1, round(proporcion * TARGET_TOTAL))
        
        # Muestreo uniforme: tomar elementos a intervalos regulares
        step = len(items) / cant_objetivo
        items_submuestra = [items[int(i * step)] for i in range(cant_objetivo)]
        
        registros_seleccionados.extend(items_submuestra)
        print(f"  • {cat}: {len(items)} originales ➔ {len(items_submuestra)} seleccionadas ({proporcion*100:.1f}%)")

    # Ajustar a exactamente 100 elementos si hubo un pequeño desbalance por redondeo
    if len(registros_seleccionados) > TARGET_TOTAL:
        registros_seleccionados = registros_seleccionados[:TARGET_TOTAL]

    dataset_tuple = tuple(registros_seleccionados)

    print("\n" + "=" * 60)
    print(f"✓ Tupla mini-mini recreada con éxito. Total: {len(dataset_tuple)} elementos.")
    print("=" * 60)

    # Mostrar los primeros 5 ejemplos
    print("\nPrimeros 5 elementos de la nueva tupla:")
    for elem in dataset_tuple[:5]:
        print(elem)

📊 Distribución original (1000 total):
  • real: 250 originales ➔ 25 seleccionadas (25.0%)
  • deepfakes: 150 originales ➔ 15 seleccionadas (15.0%)
  • face2face: 150 originales ➔ 15 seleccionadas (15.0%)
  • faceshifter: 150 originales ➔ 15 seleccionadas (15.0%)
  • faceswap: 150 originales ➔ 15 seleccionadas (15.0%)
  • neuraltextures: 150 originales ➔ 15 seleccionadas (15.0%)

✓ Tupla mini-mini recreada con éxito. Total: 100 elementos.

Primeros 5 elementos de la nueva tupla:
(0, 'real', 'd:\\OneDrive - UNIR\\Master\\99.TFM\\DataSets\\Real_mini\\real_001.jpg')
(0, 'real', 'd:\\OneDrive - UNIR\\Master\\99.TFM\\DataSets\\Real_mini\\real_011.jpg')
(0, 'real', 'd:\\OneDrive - UNIR\\Master\\99.TFM\\DataSets\\Real_mini\\real_021.jpg')
(0, 'real', 'd:\\OneDrive - UNIR\\Master\\99.TFM\\DataSets\\Real_mini\\real_031.jpg')
(0, 'real', 'd:\\OneDrive - UNIR\\Master\\99.TFM\\DataSets\\Real_mini\\real_041.jpg')


In [3]:
import json

# Guardar la tupla en un archivo JSON
with open("datasetTuplesHive.json", "w", encoding="utf-8") as f:
        json.dump(dataset_tuple, f, ensure_ascii=False, indent=4)

In [ ]:
import requests
import json
import os
import time

# Clave API de Hive
API_KEY = ''

headers = {
    'authorization': f'Bearer {API_KEY}',
}

# Archivos de entrada y salida
INPUT_DATASET = "datasetTuplesHive.json"
OUTPUT_JSON = "respuestasHive.json"

# 1. Cargar las rutas desde datasetTuplesHive.json
if os.path.exists(INPUT_DATASET):
    with open(INPUT_DATASET, "r", encoding="utf-8") as f:
        dataset_tuple = json.load(f)
    print(f"✓ Cargadas {len(dataset_tuple)} imágenes desde '{INPUT_DATASET}'")
else:
    print(f"❌ Error: No se encontró el archivo '{INPUT_DATASET}'.")
    dataset_tuple = []

# 2. Reanudar desde 'respuestasHive.json' si ya existe para evitar llamadas repetidas
if os.path.exists(OUTPUT_JSON):
    with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
        respuestas_hive = json.load(f)
    print(f"✓ Reanudando desde '{OUTPUT_JSON}' ({len(respuestas_hive)} respuestas cargadas)")
else:
    respuestas_hive = [None for _ in range(len(dataset_tuple))]

# Ajustar tamaño si hay desajustes
if len(respuestas_hive) < len(dataset_tuple):
    respuestas_hive.extend([None] * (len(dataset_tuple) - len(respuestas_hive)))

# 3. Procesar las imágenes de forma independiente
if len(dataset_tuple) > 0:
    print(f"\nIniciando evaluación con Hive para el dataset reducido...\n")

    for idx, item in enumerate(dataset_tuple):
        clasificacion, tipo_fake, file_path = item
        res_actual = respuestas_hive[idx]

        # Verificar si la respuesta actual es inválida (None o error)
        es_invalido_o_error = (
            res_actual is None or
            (isinstance(res_actual, dict) and ("error" in res_actual or "error_code" in res_actual))
        )

        # Omitir si ya tiene un valor válido
        if not es_invalido_o_error:
            print(f"[{idx+1}/{len(dataset_tuple)}] Omitiendo (ya procesado): {os.path.basename(file_path)}")
            continue

        print(f"[{idx+1}/{len(dataset_tuple)}] Procesando: {os.path.basename(file_path)} ({tipo_fake})")

        try:
            if not os.path.exists(file_path):
                res_hive = {"error": f"El archivo no existe en la ruta: {file_path}"}
            else:
                with open(file_path, 'rb') as f:
                    files = {'media': (os.path.basename(file_path), f, 'image/jpeg')}

                    response = requests.post(
                        'https://api.thehive.ai/api/v3/hive/ai-generated-and-deepfake-content-detection',
                        headers=headers,
                        files=files
                    )

                    if response.status_code == 200:
                        res_hive = response.json()
                    else:
                        res_hive = {"error_code": response.status_code, "message": response.text}

        except Exception as e:
            res_hive = {"error": str(e)}

        # Asignar la respuesta directamente en la lista
        respuestas_hive[idx] = res_hive

        # Guardado preventivo cada 10 iteraciones
        if (idx + 1) % 10 == 0:
            with open(OUTPUT_JSON, "w", encoding="utf-8") as out_file:
                json.dump(respuestas_hive, out_file, ensure_ascii=False, indent=4)

        time.sleep(0.1)

    print(f"\n✓ Proceso finalizado. Total de elementos: {len(respuestas_hive)}")

    # 4. Guardar los resultados en el nuevo archivo respuestasHive.json
    with open(OUTPUT_JSON, "w", encoding="utf-8") as out_file:
        json.dump(respuestas_hive, out_file, ensure_ascii=False, indent=4)
    print(f"✓ Resultados guardados en '{OUTPUT_JSON}'")

else:
    print("❌ No hay datos que procesar.")

✓ Cargadas 100 imágenes desde 'datasetTuplesHive.json'

Iniciando evaluación con Hive para el dataset reducido...

[1/100] Procesando: real_001.jpg (real)
[2/100] Procesando: real_011.jpg (real)
[3/100] Procesando: real_021.jpg (real)
[4/100] Procesando: real_031.jpg (real)
[5/100] Procesando: real_041.jpg (real)
[6/100] Procesando: real_051.jpg (real)
[7/100] Procesando: real_061.jpg (real)
[8/100] Procesando: real_071.jpg (real)
[9/100] Procesando: real_081.jpg (real)
[10/100] Procesando: real_091.jpg (real)
[11/100] Procesando: real_101.jpg (real)
[12/100] Procesando: real_111.jpg (real)
[13/100] Procesando: real_121.jpg (real)
[14/100] Procesando: real_131.jpg (real)
[15/100] Procesando: real_141.jpg (real)
[16/100] Procesando: real_151.jpg (real)
[17/100] Procesando: real_161.jpg (real)
[18/100] Procesando: real_171.jpg (real)
[19/100] Procesando: real_181.jpg (real)
[20/100] Procesando: real_191.jpg (real)
[21/100] Procesando: real_201.jpg (real)
[22/100] Procesando: real_211.jpg